In [3]:
import numpy as np
import pandas as pd

from datetime import datetime
from scipy.stats import skew 
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KDTree
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import os
print(os.listdir())
import category_encoders as ce
import optuna
from sklearn.metrics import mean_pinball_loss

import warnings
warnings.filterwarnings('ignore')

['house_price_competition_day_2_failed_trying.ipynb', 'house_price_advanced_view.ipynb', 'addr_kmeans.pkl', 'submission.csv', 'house_price_advanced_models.ipynb', 'my_model_submission.csv1', 'submission5.csv', 'my_model_submission4.csv', 'house_price', 'addr_umap.pkl', 'Day1.ipynb', 'day12.ipynb', 'titanic', 'house-prices-advanced-regression-techniques.zip', 'titanic.zip', 'my_model_submission3.csv', 'house_price_competition_view_day2.ipynb', 'Untitled.ipynb', 'my_model_submission0.csv', 'X_umap.npy', 'RD_competition_day10.ipynb', 'addr_tfidf.pkl', 'Day2 Housing Price.ipynb', 'pca_model.pkl', 'my_model_submission.csv', 'predictions.csv', 'competition_day11.ipynb', 'umap_model.pkl', 'predictions4.csv', 'my_model_submission1.csv', 'submission6.csv', 'house_price_view.ipynb', 'home-data-for-ml-course.zip', '.ipynb_checkpoints', 'home-data-for-ml-course', 'house_price.ipynb', 'predictions3.csv', 'failed_trying_with_nns_day_9.ipynb', 'not_so_bad_trying_with_lightgbm_day_8.ipynb', 'my_model_

In [68]:
train = pd.read_csv('house_price/dataset.csv')
test = pd.read_csv('house_price/test.csv')
print ("Data is loaded!")

quantitative = [f for f in train.columns if train.dtypes[f] != 'object']
quantitative.remove('sale_price')
quantitative.remove('id')
qualitative = [f for f in train.columns if train.dtypes[f] == 'object']

sns.set_style("whitegrid")
missing = train.isnull().sum()
missing = missing[missing > 0]
print(missing)

sns.set_style("whitegrid")
missing = test.isnull().sum()
missing = missing[missing > 0]
print(missing)

def dataset_fill_null(obj):
    obj['subdivision'].fillna('Unknown', inplace=True)
    obj.drop(columns=['sale_nbr'], inplace=True)
    obj['submarket'].fillna('Unknown', inplace=True)


dataset_fill_null(train)
dataset_fill_null(test)
print(train.shape)
print(test.shape)

# 构造原始地址字段
train_ID = train['id']
test_ID = test['id']
# Now drop the  'Id' colum since it's unnecessary for  the prediction process.
drop_cols=['id',#row_id,没有任何信息.
           'golf',#20万数据 198756都是0,基本没什么信息了.
           'view_rainier',#20万数据,198588都是0.
           'view_skyline',#20万数据,198517都是0.
           'view_lakesamm',#20万数据,198776都是0.
           'view_otherwater',#20万数据,198473都是0.
           'view_other',#20万数据,198833都是0.
          ]
train.drop(drop_cols, axis=1, inplace=True)
test_raw = test.drop(drop_cols, axis=1, inplace=True)

# Deleting outliers
train.reset_index(drop=True, inplace=True)
# We use the numpy fuction log1p which  applies log(1+x) to all elements of the column
# train["sale_price"] = np.log1p(train["sale_price"])
y = train.sale_price.reset_index(drop=True)


def add_knn_price_features(df, base_df=None, lat_col='latitude', lon_col='longitude',
                           target_col='target', ks=[5, 10, 20]):
    if base_df is None:
        base_df = df  # 默认自己为近邻池

    coords_query = df[[lat_col, lon_col]].values
    coords_base = base_df[[lat_col, lon_col]].values
    tree = KDTree(coords_base, metric='euclidean')

    base_targets = base_df[target_col].values
    knn_features = {}

    for k in ks:
        dists, indices = tree.query(coords_query, k=k)
        neighbor_targets = base_targets[indices]

        knn_features[f'knn_price_mean_{k}'] = neighbor_targets.mean(axis=1)
        knn_features[f'knn_price_std_{k}'] = neighbor_targets.std(axis=1)
        knn_features[f'knn_price_range_{k}'] = neighbor_targets.max(axis=1) - neighbor_targets.min(axis=1)

    for col, val in knn_features.items():
        df[col] = val

    return df


def add_radius_knn_features(df, base_df, lat_col='latitude', lon_col='longitude', target_col='target', radius=0.01, max_neighbors=100):
    coords = np.radians(base_df[[lat_col, lon_col]])
    tree = BallTree(coords, metric='haversine')  # 地理距离计算更准

    df_coords = np.radians(df[[lat_col, lon_col]])
    indices = tree.query_radius(df_coords, r=radius)

    # 每个样本找到若干邻居索引后，聚合
    agg_means, agg_stds, agg_counts = [], [], []
    base_targets = base_df[target_col].values

    for idxs in indices:
        if len(idxs) > 1:
            if len(idxs) > max_neighbors:
                idxs = idxs[:max_neighbors]
            neigh_vals = base_targets[idxs]
            agg_means.append(np.mean(neigh_vals))
            agg_stds.append(np.std(neigh_vals))
            agg_counts.append(len(idxs))
        else:
            agg_means.append(np.nan)
            agg_stds.append(np.nan)
            agg_counts.append(0)

    df['radius_knn_mean'] = agg_means
    df['radius_knn_std'] = agg_stds
    df['radius_knn_count'] = agg_counts

    return df


def add_knn_price_features_radius(df, base_df, lat_col='latitude', lon_col='longitude',
                                   target_col='sale_price', radius=0.01, max_neighbors=100):
    df = df.copy()
    
    coords_query = df[[lat_col, lon_col]].astype(np.float32).values
    coords_base = base_df[[lat_col, lon_col]].astype(np.float32).values
    targets_base = base_df[target_col].astype(np.float32).values

    tree = KDTree(coords_base, leaf_size=40, metric='euclidean')
    neighbor_indices = tree.query_radius(coords_query, r=radius)

    means, stds, ranges = [], [], []

    for i, inds in enumerate(neighbor_indices):
        # 排除自身（仅当 base_df 是 df）
        if base_df is df:
            inds = inds[inds != i]
        
        if len(inds) == 0:
            means.append(np.nan)
            stds.append(np.nan)
            ranges.append(np.nan)
        else:
            if len(inds) > max_neighbors:
                inds = inds[:max_neighbors]
            vals = targets_base[inds]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
            ranges.append(np.max(vals) - np.min(vals))

    df[f'knn_radius_mean'] = means
    df[f'knn_radius_std'] = stds
    df[f'knn_radius_range'] = ranges

    return df

def preprocess_and_encode(df, y=None, encoder_bundle=None, drop_high_card=True):
    df = df.copy()
    
    # ================= 清洗阶段 ================= #
    if encoder_bundle is None:
        # 训练阶段
        address_text = df['subdivision'].fillna('').str.lower()
        vectorizer = CountVectorizer(max_features=1000, token_pattern=r'\b\w+\b', ngram_range=(1, 2))
        address_vec = vectorizer.fit_transform(address_text)
        svd = TruncatedSVD(n_components=5, random_state=42)
        svd.fit(address_vec)
        explained = np.cumsum(svd.explained_variance_ratio_)
        print("Explained variance (first 5):", explained[:5])
        address_pca = svd.fit_transform(address_vec)
    else:
        # 推理阶段
        vectorizer = encoder_bundle['vectorizer']
        svd = encoder_bundle['svd']
        address_text = df['subdivision'].fillna('').str.lower()
        address_vec = vectorizer.transform(address_text)
        address_pca = svd.transform(address_vec)

    address_df = pd.DataFrame(address_pca, columns=[f'address_pca_{i+1}' for i in range(address_pca.shape[1])], index=df.index)
    df = pd.concat([df, address_df], axis=1)

    df['total_baths'] = df['bath_full'] + 0.75*df['bath_3qtr'] + 0.5*df['bath_half']
    df['total_value'] = df['land_val'] + df['imp_val']
    df['living_area'] = df['sqft'] + df['sqft_fbsmt']

    # 补充缺失
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    df[cat_cols] = df[cat_cols].fillna('None')
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # 日期和派生
    if 'sale_date' in df.columns:
        df['sale_date'] = pd.to_datetime(df['sale_date'])
        df['sale_year'] = df['sale_date'].dt.year
        df['sale_month'] = df['sale_date'].dt.month
        df['house_age'] = df['sale_year'] - df['year_built']
        df['reno_age'] = df['sale_year'] - df['year_reno']
        df['has_reno'] = (df['year_reno'] > 0).astype(int)
        df['land_imp_ratio'] = df['land_val'] / (df['imp_val'] + 1e-5)

    # 编码分类变量
    # 找出 object 类型列（可参与类别编码）
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    
    # 手动优先考虑的高基类列（确保存在才用）
    manual_high_card = ['sale_date','join_status', 'city', 'zoning','subdivision','submarket']
    high_card_cols = [col for col in manual_high_card if col in df.columns and df[col].dtype == 'object']
    
    # 自动补充高基类列（nunique > 50）
    for col in cat_cols:
        if col not in high_card_cols:
            try:
                n_unique = df[col].nunique()
                if n_unique > 50:
                    high_card_cols.append(col)
            except Exception as e:
                print(f"[异常] {col}: {e}")
    
    # 低基类列自动判断（nunique <= 50）
    low_card_cols = [col for col in cat_cols if col not in high_card_cols]

    print("low",low_card_cols, "high",high_card_cols)

    if encoder_bundle is None:
        target_encoder = ce.TargetEncoder()
        X_target = target_encoder.fit_transform(df[high_card_cols], y) if y is not None else pd.DataFrame(index=df.index)
    else:
        target_encoder = encoder_bundle['target_encoder']
        X_target = target_encoder.transform(df[high_card_cols]) if high_card_cols else pd.DataFrame(index=df.index)
    X_target.columns = [f"{col}_te" for col in high_card_cols]

    # 数值标准化
    num_cols = df.select_dtypes(include=[np.number]).columns
    if encoder_bundle is None:
        scaler = StandardScaler()
        X_num_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols, index=df.index)
    else:
        scaler = encoder_bundle['scaler']
        X_num_scaled = pd.DataFrame(scaler.transform(df[num_cols]), columns=num_cols, index=df.index)

    X_final_df = pd.concat([X_num_scaled, X_target], axis=1)

    if encoder_bundle is None:
        encoder_bundle = {
            'vectorizer': vectorizer,
            'svd': svd,
            'target_encoder': target_encoder,
            'scaler': scaler
        }

    return X_final_df, encoder_bundle



def add_knn_price_features(df, base_df=None, lat_col='latitude', lon_col='longitude',
                           target_col='target', ks=[5, 10, 20]):
    if base_df is None:
        base_df = df  # 默认自己为近邻池

    coords_query = df[[lat_col, lon_col]].values
    coords_base = base_df[[lat_col, lon_col]].values
    tree = KDTree(coords_base, metric='euclidean')

    base_targets = base_df[target_col].values
    knn_features = {}

    for k in ks:
        dists, indices = tree.query(coords_query, k=k)
        neighbor_targets = base_targets[indices]

        knn_features[f'knn_price_mean_{k}'] = neighbor_targets.mean(axis=1)
        knn_features[f'knn_price_std_{k}'] = neighbor_targets.std(axis=1)
        knn_features[f'knn_price_range_{k}'] = neighbor_targets.max(axis=1) - neighbor_targets.min(axis=1)

    for col, val in knn_features.items():
        df[col] = val

    return df


def add_radius_knn_features(df, base_df, lat_col='latitude', lon_col='longitude', target_col='target', radius=0.01, max_neighbors=100):
    coords = np.radians(base_df[[lat_col, lon_col]])
    tree = BallTree(coords, metric='haversine')  # 地理距离计算更准

    df_coords = np.radians(df[[lat_col, lon_col]])
    indices = tree.query_radius(df_coords, r=radius)

    # 每个样本找到若干邻居索引后，聚合
    agg_means, agg_stds, agg_counts = [], [], []
    base_targets = base_df[target_col].values

    for idxs in indices:
        if len(idxs) > 1:
            if len(idxs) > max_neighbors:
                idxs = idxs[:max_neighbors]
            neigh_vals = base_targets[idxs]
            agg_means.append(np.mean(neigh_vals))
            agg_stds.append(np.std(neigh_vals))
            agg_counts.append(len(idxs))
        else:
            agg_means.append(np.nan)
            agg_stds.append(np.nan)
            agg_counts.append(0)

    df['radius_knn_mean'] = agg_means
    df['radius_knn_std'] = agg_stds
    df['radius_knn_count'] = agg_counts

    return df


def add_knn_price_features_radius(df, base_df, lat_col='latitude', lon_col='longitude',
                                   target_col='sale_price', radius=0.01, max_neighbors=100):
    df = df.copy()
    
    coords_query = df[[lat_col, lon_col]].astype(np.float32).values
    coords_base = base_df[[lat_col, lon_col]].astype(np.float32).values
    targets_base = base_df[target_col].astype(np.float32).values

    tree = KDTree(coords_base, leaf_size=40, metric='euclidean')
    neighbor_indices = tree.query_radius(coords_query, r=radius)

    means, stds, ranges = [], [], []

    for i, inds in enumerate(neighbor_indices):
        # 排除自身（仅当 base_df 是 df）
        if base_df is df:
            inds = inds[inds != i]
        
        if len(inds) == 0:
            means.append(np.nan)
            stds.append(np.nan)
            ranges.append(np.nan)
        else:
            if len(inds) > max_neighbors:
                inds = inds[:max_neighbors]
            vals = targets_base[inds]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
            ranges.append(np.max(vals) - np.min(vals))

    df[f'knn_radius_mean'] = means
    df[f'knn_radius_std'] = stds
    df[f'knn_radius_range'] = ranges

    return df

# 应用预处理
# 对训练集（自己做自己）

X_train_raw, X_val, y_train, y_val = train_test_split(
    train, y, test_size=0.2, random_state=42
)

test_raw = test.copy()  # 或者正确读取原始测试集

X_train = add_knn_price_features(X_train_raw, base_df=X_train_raw, target_col='sale_price')
X_val = add_knn_price_features(X_val, base_df=X_train_raw, target_col='sale_price')
X_train_full = add_knn_price_features(train, base_df=train, target_col='sale_price')
# 对测试集（用训练集做 base）
test = add_knn_price_features(test_raw, base_df=X_train_raw, target_col='sale_price')

X_train = add_knn_price_features_radius(X_train, base_df=X_train_raw, target_col='sale_price')
X_val = add_knn_price_features_radius(X_val, base_df=X_train_raw, target_col='sale_price')
X_train_full = add_knn_price_features_radius(X_train_full, base_df=train, target_col='sale_price')
# 对测试集（用训练集做 base）
test = add_knn_price_features_radius(test, base_df=X_train_raw, target_col='sale_price')

X_train = X_train.drop(['sale_price'], axis=1)
X_val = X_val.drop(['sale_price'], axis=1)
X_train_full = X_train_full.drop(['sale_price'], axis=1)
X_train, encoder_bundle = preprocess_and_encode(X_train, y_train)
X_val, _ = preprocess_and_encode(X_val, encoder_bundle=encoder_bundle)
test, _ = preprocess_and_encode(test, encoder_bundle=encoder_bundle)
X_train_full, _ = preprocess_and_encode(X_train_full, y)

drop_knn_cols = [
    'knn_price_std_5', 'knn_price_range_5',
    'knn_price_std_10', 'knn_price_range_10',
    'knn_price_mean_20', 'knn_price_std_20', 'knn_price_range_20',
    'knn_radius_std', 'knn_radius_range'
]

X_train = X_train.drop(columns=drop_knn_cols, errors='ignore')
X_val = X_val.drop(columns=drop_knn_cols, errors='ignore')
X_train_full = X_train_full.drop(columns=drop_knn_cols, errors='ignore')
test = test.drop(columns=drop_knn_cols, errors='ignore')
knn_cols = [col for col in X_train.columns if 'knn' in col]
print("KNN 特征列:", knn_cols)

Data is loaded!
sale_nbr       42182
subdivision    17550
submarket       1717
dtype: int64
sale_nbr       42412
subdivision    17550
submarket       1718
dtype: int64
(200000, 46)
(200000, 45)
Explained variance (first 5): [0.09633825 0.15665848 0.18816216 0.21446865 0.23848159]
low [] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
low [] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
low [] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
Explained variance (first 5): [0.09625988 0.15664142 0.18813496 0.21431771 0.2384321 ]
low [] high ['join_status', 'city', 'zoning', 'subdivision', 'submarket', 'sale_warning']
KNN 特征列: ['knn_price_mean_5', 'knn_price_mean_10', 'knn_radius_mean']


In [69]:
import lightgbm as lgb

def train_quantile_model(X, y, quantile):
    params = {
        'objective': 'quantile',
        'alpha': quantile,
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'min_data_in_leaf': 30,
        'verbosity': -1
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y)
    return model


In [70]:
def interval_score(y_true, lower, upper, alpha=0.1):
    """
    Compute Wα for a given prediction interval [lower, upper] and true values y.
    """
    interval_width = upper - lower
    below = y_true < lower
    above = y_true > upper
    inside = (lower <= y_true) & (y_true <= upper)
    
    penalty = np.zeros_like(y_true, dtype=float)
    penalty[below] = (2 / alpha) * (lower[below] - y_true[below])
    penalty[above] = (2 / alpha) * (y_true[above] - upper[above])

    return np.mean(interval_width + penalty)


In [71]:
model_lower = train_quantile_model(X_train, y_train, quantile=0.05)
model_upper = train_quantile_model(X_train, y_train, quantile=0.95)

pred_lower = model_lower.predict(X_val)
pred_upper = model_upper.predict(X_val)

score = interval_score(y_val, pred_lower, pred_upper, alpha=0.1)
print(f"W_alpha score: {score:.5f}")

W_alpha score: 410623.64276


In [72]:
coverage = np.mean((y_val >= pred_lower) & (y_val <= pred_upper))
avg_width = np.mean(pred_upper - pred_lower)
print(f"Coverage: {coverage:.4f}, Width: {avg_width:.2f}")

Coverage: 0.8391, Width: 222856.90


In [73]:
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):

    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower
    penalty_lower = 2 / alpha * (lower - y_true)
    penalty_upper = 2 / alpha * (y_true - upper)

    score = width.copy()
    score += np.where(y_true < lower, penalty_lower, 0)
    score += np.where(y_true > upper, penalty_upper, 0)

    if return_coverage:
        inside = (y_true >= lower) & (y_true <= upper)
        coverage = np.mean(inside)
        return np.mean(score), coverage

    return np.mean(score)

print(winkler_score(y_val, pred_lower, pred_upper, alpha=0.1, return_coverage=True))

(410623.6427604977, 0.83915)


In [74]:
def best_features(model, num=50):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    print(feat_imp.head(num))


best_features(model_lower)
best_features(model_upper)

              feature  importance
42          sale_year        2459
31   knn_price_mean_5        1843
40        total_value        1418
2           longitude        1339
1            latitude        1269
9            sqft_lot        1159
44          house_age        1132
33    knn_radius_mean        1010
11             sqft_1         978
10               sqft         944
53    sale_warning_te         933
32  knn_price_mean_10         929
7          year_built         868
5            land_val         805
51     subdivision_te         798
43         sale_month         796
47     land_imp_ratio         736
41        living_area         710
6             imp_val         696
45           reno_age         609
22          gara_sqft         576
38      address_pca_5         563
50          zoning_te         539
3                area         527
35      address_pca_2         499
36      address_pca_3         493
48     join_status_te         478
37      address_pca_4         473
34      addres

In [15]:
top_features = [
    'sale_year', 'knn_price_mean_5', 'total_value', 'latitude', 'longitude',
    'house_age', 'sqft_lot', 'sqft', 'sale_warning_te', 'knn_radius_mean',
    'sqft_1', 'year_built', 'knn_price_mean_10', 'sale_month', 'land_val',
    'land_imp_ratio', 'imp_val', 'living_area', 'address_pca_5', 'reno_age',
    'area', 'zoning_te', 'gara_sqft', 'address_pca_2', 'address_pca_4',
    'address_pca_3', 'address_pca_1', 'grade', 'condition', 'sqft_fbsmt'
]

X_train_A = X_train[top_features]
X_val_A = X_val[top_features]
X_test_A = test[top_features]

In [34]:
X_train.drop(columns=['has_reno'], axis=1, inplace=True)
X_val.drop(columns=['has_reno'], axis=1, inplace=True)
X_train.shape

(160000, 49)

In [41]:
def train_quantile_model(X, y, quantile):
    params = {
        'objective': 'quantile',
        'alpha': quantile,
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'min_data_in_leaf': 30,
        'verbosity': -1
    }
    #best_params = {'n_estimators': 3000, 'learning_rate': 0.019873365074989813, 'num_leaves': 51, 'min_data_in_leaf': 90, 'feature_fraction': 0.8178805506274508, 'bagging_fraction': 0.789642957230141, 'bagging_freq': 2, 'lambda_l1': 4.942955084287727, 'lambda_l2': 1.968136861806871}
    #params.update(best_params)
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y)
    return model

model_lower = train_quantile_model(X_train, y_train, quantile=0.05)
model_upper = train_quantile_model(X_train, y_train, quantile=0.95)

pred_lower = model_lower.predict(X_val)
pred_upper = model_upper.predict(X_val)



In [42]:
print(winkler_score(y_val, pred_lower, pred_upper, alpha=0.1, return_coverage=True))
def best_features(model):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    print(feat_imp.head(49))
best_features(model_lower)
best_features(model_upper)

(418783.4505363162, 0.83945)
              feature  importance
42          sale_year        2355
31   knn_price_mean_5        1828
40        total_value        1532
1            latitude        1451
2           longitude        1447
44          house_age        1259
9            sqft_lot        1243
10               sqft        1049
47    sale_warning_te        1038
33    knn_radius_mean        1030
11             sqft_1        1022
7          year_built         980
32  knn_price_mean_10         911
43         sale_month         908
5            land_val         838
46     land_imp_ratio         819
6             imp_val         775
41        living_area         722
38      address_pca_5         656
45           reno_age         642
3                area         600
48          zoning_te         593
22          gara_sqft         584
35      address_pca_2         544
37      address_pca_4         539
36      address_pca_3         538
34      address_pca_1         494
13              gra

In [75]:
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder,LabelEncoder
from pathlib import Path
from sklearn import preprocessing
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.base import clone
import joblib 
import warnings
warnings.filterwarnings("ignore")


import matplotlib.pyplot as plt
print("ok")

#  训练模型
def train_quantile_model(alpha,mdoel_name):
    if mdoel_name=='cat':
        cat_params = {
            'objective': f'Quantile:alpha={alpha}',             # 回归任务，使用均方根误差
            'learning_rate': 0.05,           # 学习率
            'iterations': 8000,              # 树的数量
            'random_seed': 42,               # 随机种子
            'verbose': 800,                   # 显示训练过程
            'grow_policy' :"Depthwise",
            'min_data_in_leaf': 1000,
            'l2_leaf_reg': 100,
            'od_type':"IncToDec",
            'od_pval':0.1,
        }
        model = CatBoostRegressor(**cat_params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
        )
        return model

    elif mdoel_name=='lgbm':
        lgb_params = {
        'objective': 'quantile',       # 回归任务
        'learning_rate': 0.05,           # 学习率
        'subsample': 0.7,                # 训练实例的子采样比例
        'num_leaves': 50,                # 叶子节点数（LightGBM 特有参数）
        'n_estimators': 3000,            # 树的数量
        'random_state': 42,              # 随机种子
        'alpha':alpha,
        'subsample_freq':1,
        #'colsample_bytree':0.5,

        #'verbose': 0                  # 显示训练过程
    }
        model = lgb.LGBMRegressor(**lgb_params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            #eval_metric='quantile',
            #callbacks=[lgb.early_stopping(stopping_rounds=5000, verbose=False)]
        )
        return model


print("ok")

#  训练模型
def train_quantile_model(alpha,mdoel_name):
    if mdoel_name=='cat':
        cat_params = {
            'objective': f'Quantile:alpha={alpha}',             # 回归任务，使用均方根误差
            'learning_rate': 0.05,           # 学习率
            'iterations': 8000,              # 树的数量
            'random_seed': 42,               # 随机种子
            'verbose': 800,                   # 显示训练过程
            'grow_policy' :"Depthwise",
            'min_data_in_leaf': 1000,
            'l2_leaf_reg': 100,
            'od_type':"IncToDec",
            'od_pval':0.1,
        }
        model = CatBoostRegressor(**cat_params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
        )
        return model

    elif mdoel_name=='lgbm':
        lgb_params = {
        'objective': 'quantile',       # 回归任务
        'learning_rate': 0.05,           # 学习率
        'subsample': 0.7,                # 训练实例的子采样比例
        'num_leaves': 50,                # 叶子节点数（LightGBM 特有参数）
        'n_estimators': 3000,            # 树的数量
        'random_state': 42,              # 随机种子
        'alpha':alpha,
        'subsample_freq':1,
        #'colsample_bytree':0.5,

        #'verbose': 0                  # 显示训练过程
    }
        model = lgb.LGBMRegressor(**lgb_params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            #eval_metric='quantile',
            #callbacks=[lgb.early_stopping(stopping_rounds=5000, verbose=False)]
        )
        return model


print("ok")

cat_model_lower = train_quantile_model(0.05,"cat")
print("cat_model_lower is ok")
cat_model_upper = train_quantile_model(0.95,"cat")
print('cat_model_upper is ok')
print("cat ok")


#lgbm_model_lower = train_quantile_model(0.05,"lgbm")
#print("lgbm_model_lower is ok")
#lgbm_model_upper = train_quantile_model(0.95,"lgbm")
#print('lgbm_model_upper is ok')
#print("lgbm ok")

cat_lower = cat_model_lower.predict(X_val)
cat_upper = cat_model_upper.predict(X_val)

#lgbm_lower = lgbm_model_lower.predict(X_val)
#lgbm_upper = lgbm_model_upper.predict(X_val)
print("ok")

ok
ok
ok
0:	learn: 21183.8578391	test: 21148.6696421	best: 21148.6696421 (0)	total: 28ms	remaining: 3m 43s
800:	learn: 6861.7968714	test: 7830.6820708	best: 7830.6820708 (800)	total: 20.2s	remaining: 3m 1s
1600:	learn: 6476.5517668	test: 7707.0349237	best: 7707.0349237 (1600)	total: 41.5s	remaining: 2m 45s
2400:	learn: 6284.9634681	test: 7666.3173356	best: 7666.1109196 (2390)	total: 1m 1s	remaining: 2m 23s
3200:	learn: 6157.3611478	test: 7644.3433098	best: 7644.2645584 (3195)	total: 1m 21s	remaining: 2m 2s
4000:	learn: 6061.5538626	test: 7630.4806033	best: 7630.4006067 (3994)	total: 1m 41s	remaining: 1m 41s
4800:	learn: 5983.2905398	test: 7625.0666201	best: 7624.4436850 (4748)	total: 2m 1s	remaining: 1m 20s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 7624.443685
bestIteration = 4748

Shrink model to first 4749 iterations.
cat_model_lower is ok
0:	learn: 62520.8803947	test: 63028.6442538	best: 63028.6442538 (0)	total: 21.4ms	remaining: 2m 51s
800:	learn: 7803.36108

In [76]:
cat_score=winkler_score(y_val,cat_lower,cat_upper)
#lgbm_score=winkler_score(y_val,lgbm_lower,lgbm_upper)

print(f"cat :{cat_score}")
#print(f"lgbm :{lgbm_score}")


cat_score=winkler_score(y_val,cat_lower-15000,cat_upper+15000)
#lgbm_score=winkler_score(y_val,lgbm_lower-25000,lgbm_upper+25000)

print(f"cat :{cat_score}")
#print(f"lgbm :{lgbm_score}")

cat_score=winkler_score(y_val,cat_lower-7500,cat_upper+7500)
#lgbm_score=winkler_score(y_val,lgbm_lower-15000,lgbm_upper+15000)

print(f"cat :{cat_score}")
#print(f"lgbm :{lgbm_score}")

cat :380815.188204244
cat :374136.61096335074
cat :374986.72376958944


In [77]:
best_features(cat_model_lower)
best_features(cat_model_upper)

              feature  importance
42          sale_year   25.824550
40        total_value   15.406592
31   knn_price_mean_5   13.270915
45           reno_age   11.269275
32  knn_price_mean_10    3.734219
13              grade    3.326936
48     join_status_te    2.688993
33    knn_radius_mean    2.153322
5            land_val    1.904710
10               sqft    1.815863
53    sale_warning_te    1.793706
6             imp_val    1.750989
44          house_age    1.629659
51     subdivision_te    1.493520
49            city_te    1.387294
52       submarket_te    1.203944
41        living_area    0.867736
1            latitude    0.760626
14        fbsmt_grade    0.672298
39        total_baths    0.640252
0           join_year    0.632008
50          zoning_te    0.603989
2           longitude    0.493675
7          year_built    0.456709
12         sqft_fbsmt    0.409884
9            sqft_lot    0.306563
11             sqft_1    0.296682
8           year_reno    0.273506
15          co

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_pinball_loss, make_scorer

param_grid = {
    'num_leaves': [4, 8, 16],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [1000, 2000],
    'feature_fraction': [0.6, 0.8],
    'bagging_fraction': [0.6, 0.8],
    'min_data_in_leaf': [20, 30],
}

def pinball_scorer(y_true, y_pred):
    return -mean_pinball_loss(y_true, y_pred, alpha=0.05)

scorer = make_scorer(pinball_scorer, greater_is_better=True)


model = lgb.LGBMRegressor(objective='quantile', alpha=0.05)
grid = GridSearchCV(model, param_grid, cv=3, scoring=scorer)
grid.fit(X, y)

print(grid.best_params_)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

params = {
    'objective': 'quantile',
    'alpha': 0.05,  # 下限模型
    'num_leaves': 16,
    'learning_rate': 0.05,
    'n_estimators': 2000,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.6,
    'min_data_in_leaf': 30,
    'verbosity': -1
}

model_lower = lgb.LGBMRegressor(**params)
model_lower.fit(X_train, y_train)

# 上限只改 alpha
params['alpha'] = 0.95
model_upper = lgb.LGBMRegressor(**params)
model_upper.fit(X_train, y_train)

pred_lower = model_lower.predict(X_val)
pred_upper = model_upper.predict(X_val)

score = interval_score(y_val, pred_lower, pred_upper, alpha=0.1)
print(f"W_alpha score: {score:.5f}")

In [ ]:
pred_lower = model_lower.predict(X_val)
pred_upper = model_upper.predict(X_val)

score = interval_score(y_val, pred_lower, pred_upper, alpha=0.1)
print(f"W_alpha score: {score:.5f}")

In [ ]:
def try_param_set(X, y, param_overrides):
    base_params = {
        'objective': 'quantile',
        'alpha': 0.05,
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'min_data_in_leaf': 30,
        'verbosity': -1
    }
    base_params.update(param_overrides)
    model = lgb.LGBMRegressor(**base_params)
    scores = cv_rmse(model, X)
    return np.mean(scores), param_overrides


In [ ]:
for leaves in [4, 8, 16]:
    for lr in [0.01, 0.05]:
        score, params = try_param_set(X, y, {'num_leaves': leaves, 'learning_rate': lr})
        print(score, params)


In [ ]:
other_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 4,
    'max_features': 'sqrt',
    'min_samples_leaf': 15,
    'min_samples_split': 10,
    'random_state': 42
}

gbr_lower = GradientBoostingRegressor(loss='quantile', alpha=0.05, **other_params)
gbr_upper = GradientBoostingRegressor(loss='quantile', alpha=0.95, **other_params)

gbr_lower.fit(X_train, y_train)
gbr_upper.fit(X_train, y_train)

pred_lower = gbr_lower.predict(X_val)
pred_upper = gbr_upper.predict(X_val)

# 评估 W_alpha
score = interval_score(y_val, pred_lower, pred_upper, alpha=0.1)
print(f"W_alpha score: {score:.5f}")

In [ ]:
best_features(model_lower)
best_features(model_upper)

In [ ]:
errors = (y_val < pred_lower) | (y_val > pred_upper)
error_cases = X_val[errors]
error_cases['sale_year']

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def analyze_top_features_loss(X_val, y_val, pred_lower, pred_upper, feature_importance, top_n=10):
    # 1. 前 top N 特征
    top_feats = feature_importance['feature'].iloc[:top_n].tolist()
    
    # 2. 计算每行 W_alpha 损失
    alpha = 0.1
    interval_width = pred_upper - pred_lower
    below = (y_val < pred_lower).astype(int)
    above = (y_val > pred_upper).astype(int)
    loss = interval_width + (2 / alpha) * (below + above)

    df_loss = X_val[top_feats].copy()
    df_loss['loss'] = loss

    # 3. 分组统计并画图
    for feat in top_feats:
        df_loss['group'] = pd.qcut(df_loss[feat], q=10, duplicates='drop')
        plt.figure(figsize=(10, 4))
        sns.boxplot(x='group', y='loss', data=df_loss)
        plt.xticks(rotation=45)
        plt.title(f'W(alpha=0.1) Loss by {feat}')
        plt.tight_layout()
        plt.show()


In [ ]:
def best_features(model):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    return (feat_imp.head(30))


feat_imp_df = best_features(model_lower)
feat_imp_df1 = best_features(model_upper)

In [ ]:
analyze_top_features_loss(
    X_val=X_val,
    y_val=y_val,
    pred_lower=pred_lower,
    pred_upper=pred_upper,
    feature_importance=feat_imp_df,
    top_n=10
)


In [ ]:
def best_features(model):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    return (feat_imp.head(30))


best_features(model_lower)
best_features(model_upper)


In [ ]:
[k for k in X.columns if 'knn_price' in k]

In [ ]:
def add_knn_price_features(df, lat_col='latitude', lon_col='longitude', target_col='target', ks=[5, 10, 20]):
    coords = df[[lat_col, lon_col]].values
    target = df[target_col].values
    tree = KDTree(coords, metric='euclidean')

    # 用于保存所有新特征
    knn_features = {}

    for k in ks:
        # 找最近邻（包括自己）
        dists, indices = tree.query(coords, k=k+1)  # k+1 是因为自己也算在内

        # 排除自身
        neighbor_targets = np.array([target[idxs[1:]] for idxs in indices])

        # 聚合统计
        knn_features[f'knn_price_mean_{k}'] = neighbor_targets.mean(axis=1)
        knn_features[f'knn_price_std_{k}'] = neighbor_targets.std(axis=1)
        knn_features[f'knn_price_range_{k}'] = neighbor_targets.max(axis=1) - neighbor_targets.min(axis=1)

    # 加入原始 df
    for col_name, values in knn_features.items():
        df[col_name] = values

    return df 


In [ ]:
# 假设 df 中已经有 latitude、longitude、target（真实价格）列
df = add_knn_price_features(train, target_col='sale_price', ks=[5, 10, 20])


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_knn_feature(df, feature_col='knn_price_mean_5'):
    plt.figure(figsize=(10, 8))
    sc = plt.scatter(df['longitude'], df['latitude'], 
                     c=df[feature_col], cmap='viridis', s=5)
    plt.colorbar(sc, label=feature_col)
    plt.title(f'Spatial distribution of {feature_col}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True)
    plt.show()


In [ ]:
plot_knn_feature(df, 'knn_price_mean_20')
plot_knn_feature(df, 'knn_price_mean_10')
plot_knn_feature(df, 'knn_price_mean_5')

In [ ]:

df = train.copy()
addresses = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()

# 分词（空格切）
# 用 CountVectorizer 可以保留频率稀疏性

vectorizer = CountVectorizer(
    max_features=1000,         # 可调大小
    stop_words=None,      # 去常见词
    token_pattern=r'\b\w+\b',  # 标准单词
    ngram_range=(1, 2)         # 一元和二元组都试试
)
address_vecs = vectorizer.fit_transform(addresses)

# 得到 token -> index 映射
tokens = vectorizer.get_feature_names_out()
# 每个 token 的总出现频率（在所有样本中出现的总次数）
token_freq = np.asarray(address_vecs.sum(axis=0)).ravel()

# 排序
sorted_idx = np.argsort(-token_freq)
top_tokens = [(tokens[i], token_freq[i]) for i in sorted_idx[:200]]

# 展示前 50 个 token 和它们的频率
for t, f in top_tokens:
    print(f"{t}: {f}")
svd = TruncatedSVD(n_components=100, random_state=42)
address_pca = svd.fit_transform(address_vecs)

# 累计解释方差
explained = np.cumsum(svd.explained_variance_ratio_)
plt.plot(range(1, len(explained) + 1), explained)
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Truncated SVD on Address')
plt.grid(True)
plt.show()


In [ ]:
待处理特征：
改造时间距离现在多少年
地址经纬度做近邻，最好找出价格最高的中心点
尝试销售年份和submarke捆绑，销售月份和地区捆绑
PCA提取地址主成分

节假日等


In [ ]:
尝试模型：
NN：TabNet / DNN


In [ ]:
Ensemble 评估（集成）

In [ ]:
import numpy as np
import pandas as pd

from datetime import datetime
from scipy.stats import skew 
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KDTree
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import category_encoders as ce


import os
print(os.listdir())

import warnings
warnings.filterwarnings('ignore')